## Within-Participant Variability

### Imports

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.linear_model import LinearRegression
from scipy.optimize import curve_fit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import spearmanr
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import find_peaks, medfilt
from scipy.interpolate import make_interp_spline
from sklearn.model_selection import train_test_split
from sklearn.model_selection import  KFold, GroupKFold

from model_functions import *
from helper_functions import *

In [ ]:
def within_subject_variability(exclude_ids=None):
    CSV_FEATURES = r"PATH_TO_CSV.csv"
    CSV_METADATA = r"PATH_TO_METADATA.csv"

    features, amplitude, speed, trial_names, trial_numbers, handedness, motor_skills = load_speed_amp(CSV_FEATURES, CSV_METADATA, None, exclude_ids=exclude_ids)
    ids = features["ids"].values

    results = []
    for pid in np.unique(ids):
        pid_mask = (ids == pid)
        amp_p = amplitude[pid_mask]
        spd_p = speed[pid_mask]

        mean_amp = np.mean(amp_p)
        std_amp = np.std(amp_p)
        cv_amp = std_amp / mean_amp

        mean_spd = np.mean(spd_p)
        std_spd = np.std(spd_p)
        cv_spd = std_spd / mean_spd

        results.append({"participant": f"P{int(pid):03d}",
                        "mean_amp": mean_amp, "std_amp": std_amp, "cv_amp": cv_amp,
                        "mean_spd": mean_spd, "std_spd": std_spd, "cv_spd": cv_spd,})

    df = pd.DataFrame(results)

    print("=== Within-Subject Variability ===\n")
    print(df[["participant", "mean_amp", "std_amp", "cv_amp",
              "mean_spd", "std_spd", "cv_spd"]].to_string(index=False, float_format="{:.4f}".format))

    print(f"\n--- Aggregated ---")
    print(f"CV Amplitude: mean={df['cv_amp'].mean():.4f}, std={df['cv_amp'].std():.4f}, "
          f"min={df['cv_amp'].min():.4f}, max={df['cv_amp'].max():.4f}")
    print(f"CV Speed:     mean={df['cv_spd'].mean():.4f}, std={df['cv_spd'].std():.4f}, "
          f"min={df['cv_spd'].min():.4f}, max={df['cv_spd'].max():.4f}")

    corr_amp, p_amp = spearmanr(df["mean_amp"], df["std_amp"])
    corr_spd, p_spd = spearmanr(df["mean_spd"], df["std_spd"])

    print(f"\n--- Heteroscedasticity (Spearman: mean vs std) ---")
    print(f"Amplitude: r={corr_amp:.4f}, p={p_amp:.4f} "
          f"({'heteroscedastic' if p_amp < 0.05 else 'no evidence of heteroscedasticity'})")
    print(f"Speed:     r={corr_spd:.4f}, p={p_spd:.4f} "
          f"({'heteroscedastic' if p_spd < 0.05 else 'no evidence of heteroscedasticity'})")

    return df

In [ ]:
within_subject_variability()